**Common Prompt Chaining Patterns**

Sequential chaining: Step-by-step refinement (e.g., draft → improve → summarize).
Branching chaining: Different prompts depending on conditions (e.g., sentiment → positive vs negative flow).
Parallel chaining: Multiple prompts run independently, then merged.

In [1]:
from openai import OpenAI
client = OpenAI()

**Sequential chaining (Summarize → Extract Keywords → Generate Title)**

In [2]:
def summarize(text):
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role":"user","content":f"Summarize in 3 sentences:\n{text}"}]
    )
    return resp.choices[0].message.content

def extract_keywords(summary):
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role":"user","content":f"Extract 5 keywords from:\n{summary}"}]
    )
    return resp.choices[0].message.content

def generate_title(keywords):
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role":"user","content":f"Create a catchy title using:\n{keywords}"}]
    )
    return resp.choices[0].message.content

text = "AWS cloud architecture provides scalability, cost efficiency, and strong security compliance for enterprises."
summary = summarize(text)
keywords = extract_keywords(summary)
title = generate_title(keywords)

print("Summary:", summary)
print("Keywords:", keywords)
print("Title:", title)

Summary: AWS cloud architecture offers businesses the ability to scale resources up or down based on demand, ensuring they only pay for what they use. Its cost-efficient model helps enterprises manage budgets effectively while taking advantage of flexible pricing options. Additionally, AWS prioritizes strong security compliance, enabling organizations to protect their data and meet regulatory requirements.
Keywords: 1. AWS
2. Cloud Architecture
3. Scalability
4. Cost-efficient
5. Security Compliance
Title: "Mastering AWS Cloud Architecture: Unleashing Scalability with Cost-Efficiency and Security Compliance"


**Branching Chaining (Sentiment → Conditional Response)**

In [4]:
def sentiment(text):
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role":"user","content":f"Classify sentiment as Positive, Neutral, or Negative:\n{text}"}]
    )
    return resp.choices[0].message.content.strip()

def follow_up(text):
    s = sentiment(text)
    if "Negative" in s:
        prompt = "Suggest improvements to address concerns:\n" + text
    else:
        prompt = "Highlight strengths and opportunities:\n" + text
    
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role":"user","content":prompt}]
    )
    return {"sentiment": s, "analysis": resp.choices[0].message.content}

# Run branching chain
input_text = "The AWS setup is too expensive and hard to manage."
result = follow_up(input_text)
print(result);

{'sentiment': 'Negative', 'analysis': 'Addressing concerns about an expensive and hard-to-manage AWS setup requires a multi-faceted approach. Here are several suggestions for improvements:\n\n### Cost Management Improvements\n\n1. **Right-Sizing Resources**:\n   - Evaluate current resource utilization and scale down instances or services that are over-provisioned. Use AWS Cost Explorer and Trusted Advisor for insights.\n   - Implement auto-scaling to ensure resources only scale up when needed.\n\n2. **Use Reserved Instances or Savings Plans**:\n   - Purchase Reserved Instances or Savings Plans for predictable workloads to reduce costs significantly compared to on-demand pricing.\n\n3. **Implement Spot Instances**:\n   - Utilize Amazon EC2 Spot Instances for non-time-sensitive workloads as they can offer significant savings compared to on-demand instances.\n\n4. **Optimize Storage**:\n   - Move infrequently accessed data to lower-cost storage solutions like Amazon S3 Glacier or S3 Infre

**Parallel Chaining (Run Multiple Prompts Independently)**

In [5]:
def parallel_analysis(text):
    summary = summarize(text)
    sentiment_result = sentiment(text)
    keywords = extract_keywords(summary)
    return {
        "summary": summary,
        "sentiment": sentiment_result,
        "keywords": keywords
    }

# Run parallel chain
input_text = "AWS cloud architecture provides scalability, cost efficiency, and strong security compliance for enterprises."
result = parallel_analysis(input_text)
print(result)

{'summary': 'AWS cloud architecture enables enterprises to easily scale their resources to meet demand, ensuring they can grow without significant upfront costs. It offers cost efficiency through pay-as-you-go pricing models, allowing businesses to optimize their expenditures based on actual usage. Additionally, AWS adheres to stringent security compliance standards, further enhancing the protection of sensitive data and applications in the cloud.', 'sentiment': 'Positive', 'keywords': '1. AWS\n2. Cloud architecture\n3. Scalability\n4. Cost efficiency\n5. Security compliance'}
